# Int8-counter fine-tuning of **Ternary Bonsai 2 9B** on agent data (Colab A100 or G4)

The model stays ternary the whole time: 2-bit codes packed exactly as in PrismML's `PQ2_0` GGUF, fixed fp16 scales,
plus one **int8 counter per weight** (≈ 1.25 bytes of training state per weight, no master weights, no optimizer
state, no full-model gradient: each layer's counter is updated inside its own backward pass). The result is written
back into the GGUF, so it runs in PrismML's llama.cpp on a CPU.

Counter setting = the one tuned in the paper: τ 32, vote 0.1, leak 0.003, one trajectory per step.
Data: `openbmb/UltraData-SFT-Agent-2609`, subsets Tool-Use + Code-Agent (streamed, a few thousand trajectories),
loss only on assistant turns.

**Checks before any training (the notebook stops if one fails):**
1. *Tiny model* (~3 min): a random model with the same architecture and Hadamard folding; our PyTorch port must give
   the same perplexity (within 0.1 %, against a float32 copy of the same weights) and the same greedy tokens as PrismML's llama.cpp; a 30-step training run must
   flip weights, and the exported GGUF must load in llama.cpp and match.
2. *Real model* (~10 min): our port vs llama.cpp on WikiText (within 3 %), tokenizer identical, one training step
   fits in memory.

**Licence warning.** The dataset card says public access "is not a grant of redistribution, commercial use, or
training rights". Training here is for research; get written permission from OpenBMB before publishing the model.

Runtime → Change runtime type → **A100** (or G4) → Run all. Everything is saved in `My Drive/bonsai_counter/`.

## Setup (~8 min: packages + building PrismML's llama.cpp for CPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
W = '/content/drive/MyDrive/bonsai_counter'
os.makedirs(f'{W}/runs', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers datasets huggingface_hub flash-linear-attention
if not os.path.exists('/content/llama.cpp'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'prism', 'https://github.com/PrismML-Eng/llama.cpp',
                    '/content/llama.cpp'], check=True)
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=OFF -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /content/cmake.log 2>&1 && echo configured
!cmake --build /content/llama.cpp/build -j --target llama-perplexity llama-simple llama-tokenize > /content/build.log 2>&1 && echo built || tail -20 /content/build.log
sys.path[:0] = ['/content/llama.cpp/gguf-py', '/content']

In [ ]:
%%writefile /content/bonsai_counter.py
"""Int8-counter fine-tuning of Ternary Bonsai 2 (PrismML, Qwen3.5 hybrid architecture) straight from its PQ2_0 GGUF.

The model is a PyTorch port of the graph in PrismML's llama.cpp fork (src/models/qwen35.cpp, branch `prism`):
- weights stay ternary: 2-bit codes (-1/0/+1) packed 4 per byte exactly as in PQ2_0, plus the fp16 scale of each
  group of 128 weights, which is kept fixed;
- Hadamard-folded weights: y = W_stored @ H_b(signs * perm(x)), H_b = block-diagonal normalized Sylvester Hadamard;
- each trainable layer has an int8 counter; the update runs inside the layer's backward pass, so no gradient is
  ever stored for the whole model (state = 1 B counter + 0.25 B packed code per weight);
- after training the codes are written back into a copy of the GGUF (same block layout), so the result runs in
  PrismML's llama.cpp on a CPU.
"""
import argparse, json, math, os, re, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

PQ2_0 = 142            # ggml type id in PrismML's fork
QK = 128               # weights per PQ2_0 block
SHIFTS = None


# ----------------------------------------------------------------------------- GGUF helpers
def fval(reader, key, default=None):
    f = reader.fields.get(key)
    if f is None:
        return default
    v = f.contents()
    return v


def unpack(packed):
    """uint8 [out, in/4] -> int8 codes [out, in] in {-1, 0, +1} (PQ2_0 bit order: element j -> byte j//4, bits 2*(j%4))."""
    global SHIFTS
    if SHIFTS is None or SHIFTS.device != packed.device:
        SHIFTS = torch.tensor([0, 2, 4, 6], dtype=torch.uint8, device=packed.device)
    q = (packed.unsqueeze(-1) >> SHIFTS) & 3
    return q.reshape(packed.shape[0], -1).to(torch.int8) - 1


def pack(codes):
    v = (codes + 1).to(torch.uint8).reshape(codes.shape[0], -1, 4)
    return v[..., 0] | (v[..., 1] << 2) | (v[..., 2] << 4) | (v[..., 3] << 6)


def pq2_split(raw, out_f, in_f):
    """raw block bytes [out, in/128 * 34] -> (packed uint8 [out, in/4], scales fp16 [out, in/128])."""
    b = raw.reshape(out_f, in_f // QK, 2 + QK // 4)
    scales = np.ascontiguousarray(b[..., :2]).view(np.float16).reshape(out_f, in_f // QK)
    packed = np.ascontiguousarray(b[..., 2:]).reshape(out_f, in_f // 4)
    return packed, scales


def hadamard(b, device):
    i = torch.arange(b, device=device)
    x = i[:, None] & i[None, :]
    par = torch.zeros_like(x)
    while bool(x.any()):
        par ^= x & 1
        x = x >> 1
    return (1 - 2 * par).float() / math.sqrt(b)


class Rot:
    """Input-side transform of a Hadamard-folded weight: H_b(signs * perm(x)) (signs then H, as build_lora_mm)."""
    def __init__(self, H, signs=None, perm=None, inverse=False):
        self.H, self.signs, self.perm, self.inverse = H, signs, perm, inverse

    def __call__(self, x):
        dt, shp, b = x.dtype, x.shape, self.H.shape[0]
        x = x.float()
        if self.inverse:                         # embedding rows: h = signs * (H z)
            x = (x.reshape(*shp[:-1], -1, b) @ self.H).reshape(shp)
            return (x * self.signs if self.signs is not None else x).to(dt)
        if self.perm is not None:                # tiled [hd, nk, rep] -> grouped [hd, rep, nk] (ggml order)
            hd, nk, rep = self.perm
            x = x.reshape(*shp[:-1], rep, nk, hd).transpose(-3, -2).reshape(shp)
        if self.signs is not None:
            x = x * self.signs
        return (x.reshape(*shp[:-1], -1, b) @ self.H).reshape(shp).to(dt)


# ----------------------------------------------------------------------------- counter-trained ternary layer
class TernFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, anchor, mod):
        ctx.mod = mod
        ctx.save_for_backward(x)
        return x @ mod.weight(x.dtype).t()

    @staticmethod
    def backward(ctx, gy):
        (x,) = ctx.saved_tensors
        mod = ctx.mod
        gx = gy @ mod.weight(gy.dtype)
        if mod.trainable and mod.voting:
            g2, x2 = gy.reshape(-1, gy.shape[-1]), x.reshape(-1, x.shape[-1])
            mod.vote(g2.t().matmul(x2).float())
        return gx, torch.zeros_like(mod.anchor), None


class TLinear(nn.Module):
    def __init__(self, name, packed, scales, rot, trainable, hp):
        super().__init__()
        self.name, self.rot, self.trainable, self.hp, self.voting = name, rot, trainable, hp, False
        self.out_f, self.in_f = packed.shape[0], packed.shape[1] * 4
        self.register_buffer('packed', packed)
        self.register_buffer('scales', scales)
        self.counter = torch.zeros(self.out_f, self.in_f, dtype=torch.int8, device=packed.device) if trainable else None
        # a scalar that requires grad, so autograd reaches every layer's backward (the weights are not Parameters)
        self.anchor = nn.Parameter(torch.zeros((), device=packed.device), requires_grad=trainable)
        self.flips = 0

    def weight(self, dtype):
        c = unpack(self.packed).to(dtype).reshape(self.out_f, -1, QK)
        return (c * self.scales.to(dtype).unsqueeze(-1)).reshape(self.out_f, self.in_f)

    def forward(self, x, xr=None):
        if self.rot is not None:
            x = xr if xr is not None else self.rot(x)
        return TernFn.apply(x, self.anchor, self)

    @torch.no_grad()
    def vote(self, gW):                          # identical to the counter rule of bitnet_cpu_finetune.py
        hp = self.hp
        s = -hp['vote'] * gW / gW.pow(2).mean().sqrt().clamp(min=1e-30)
        cf = self.counter.float()
        if hp['leak']:
            cf = cf - torch.sign(cf) * (torch.rand_like(cf) < (hp['leak'] * cf.abs()).clamp(max=1)).float()
        cf = (cf + torch.floor(s + torch.rand_like(s))).clamp(-127, 127)
        del s
        up, dn = cf >= hp['tau'], cf <= -hp['tau']
        if bool((up | dn).any()):
            t = unpack(self.packed)
            nt = torch.where(up, (t + 1).clamp(max=1), torch.where(dn, (t - 1).clamp(min=-1), t))
            k = int((nt != t).sum())
            if k:
                self.packed.copy_(pack(nt)); self.flips += k
            cf = torch.where(up | dn, torch.zeros_like(cf), cf)
        self.counter.copy_(cf.to(torch.int8))


class Dense(nn.Module):                         # frozen non-ternary weight (F32/F16/BF16 or other quant, dequantized)
    def __init__(self, W, rot):
        super().__init__()
        self.register_buffer('W', W)
        self.rot = rot

    def forward(self, x, xr=None):
        if self.rot is not None:
            x = xr if xr is not None else self.rot(x)
        return x @ self.W.to(x.dtype).t()


def rms(x, w, eps):
    xf = x.float()
    return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps) * w.float()).to(x.dtype)


# ----------------------------------------------------------------------------- gated delta rule kernels
def gdn_naive(q, k, v, g, beta, scale):
    """Reference recurrence of llama.cpp build_delta_net_autoregressive. q,k,v [B,T,H,D], g,beta [B,T,H]."""
    B, T, H, Dk = q.shape
    Dv = v.shape[-1]
    S = torch.zeros(B, H, Dv, Dk, device=q.device, dtype=torch.float32)
    out = []
    q, k, v, g, beta = q.float() * scale, k.float(), v.float(), g.float(), beta.float()
    for t in range(T):
        S = S * g[:, t].exp()[..., None, None]
        sk = torch.einsum('bhvk,bhk->bhv', S, k[:, t])
        d = (v[:, t] - sk) * beta[:, t][..., None]
        S = S + torch.einsum('bhv,bhk->bhvk', d, k[:, t])
        out.append(torch.einsum('bhvk,bhk->bhv', S, q[:, t]))
    return torch.stack(out, 1)


GDN_IMPL = None


def gdn(q, k, v, g, beta, scale):
    if GDN_IMPL == 'fla':
        from fla.ops.gated_delta_rule import chunk_gated_delta_rule
        dt = q.dtype
        lo = torch.bfloat16 if dt == torch.float32 else dt          # the Triton kernels want 16-bit q, k, v
        o, _ = chunk_gated_delta_rule(q.to(lo), k.to(lo), v.to(lo), g.float(), beta.to(lo), scale=scale)
        return o.to(dt)
    if GDN_IMPL == 'hf':
        from transformers.models.qwen3_next.modeling_qwen3_next import torch_chunk_gated_delta_rule
        o, _ = torch_chunk_gated_delta_rule(q * (scale * math.sqrt(q.shape[-1])), k, v, g.float(), beta.float())
        return o.to(q.dtype)
    return gdn_naive(q, k, v, g, beta, scale).to(q.dtype)


def pick_gdn(device):
    """Choose the fastest kernel that matches the reference recurrence on random inputs."""
    global GDN_IMPL
    torch.manual_seed(0)
    B, T, H, D = 1, 96, 4, 64
    q = F.normalize(torch.randn(B, T, H, D, device=device), dim=-1).bfloat16()
    k = F.normalize(torch.randn(B, T, H, D, device=device), dim=-1).bfloat16()
    v = torch.randn(B, T, H, D, device=device).bfloat16()
    g = -F.softplus(torch.randn(B, T, H, device=device))
    beta = torch.rand(B, T, H, device=device).bfloat16()
    ref = gdn_naive(q, k, v, g, beta, D ** -0.5)
    report = {}
    for impl in ('fla', 'hf'):
        GDN_IMPL = impl
        try:
            o = gdn(q, k, v, g, beta, D ** -0.5).float()
            err = float((o - ref).abs().max() / ref.abs().max())
            report[impl] = round(err, 4)
            if err < 0.03:
                print(f'gated delta rule: using {impl} (max rel err {err:.4f} vs reference)', flush=True)
                return report
        except Exception as e:
            report[impl] = f'failed: {type(e).__name__}: {str(e)[:120]}'
    GDN_IMPL = 'naive'
    print('gated delta rule: WARNING, only the slow reference loop works:', report, flush=True)
    return report


# ----------------------------------------------------------------------------- the model
class Bonsai(nn.Module):
    def __init__(self, gguf_path, device='cuda', hp=None, dtype=torch.bfloat16):
        super().__init__()
        from gguf import GGUFReader
        from gguf.quants import dequantize
        self.hp = hp or dict(tau=32.0, vote=0.1, leak=0.003)
        r = GGUFReader(gguf_path)
        self.path, self.device = gguf_path, device
        arch = fval(r, 'general.architecture')
        assert arch in ('qwen35',), f'unsupported architecture {arch}'
        A = lambda k, d=None: fval(r, f'{arch}.{k}', d)
        self.n_embd, self.n_layer = int(A('embedding_length')), int(A('block_count'))
        self.n_head, self.n_kv = int(A('attention.head_count')), int(A('attention.head_count_kv'))
        self.hd = int(A('attention.key_length'))
        self.eps = float(A('attention.layer_norm_rms_epsilon'))
        self.n_rot, self.base = int(A('rope.dimension_count')), float(A('rope.freq_base'))
        self.sections = [int(s) for s in (A('rope.dimension_sections') or [0, 0, 0, 0])][:4]
        self.d_conv, self.d_inner = int(A('ssm.conv_kernel')), int(A('ssm.inner_size'))
        self.hk, self.nv, self.nk = int(A('ssm.state_size')), int(A('ssm.time_step_rank')), int(A('ssm.group_count'))
        self.hv = self.d_inner // self.nv
        # Hadamard contract
        hv = fval(r, 'prism.hadamard.version')
        self.had = None
        if hv is not None:
            b = int(fval(r, 'prism.hadamard.block_size'))
            assert fval(r, 'prism.hadamard.transform') == 'normalized-sylvester-walsh-hadamard'
            assert fval(r, 'prism.hadamard.axis') == 'input-last-dimension'
            H = hadamard(b, device)
            signs = {}
            if fval(r, 'prism.hadamard.sign_mode') == 'explicit':
                widths, vals, off = list(fval(r, 'prism.hadamard.sign_widths')), list(fval(r, 'prism.hadamard.sign_values')), 0
                for w in widths:
                    signs[int(w)] = torch.tensor(vals[off:off + w], dtype=torch.float32, device=device); off += w
            self.had = dict(version=int(hv), H=H, signs=signs, names=set(fval(r, 'prism.hadamard.weight_names')),
                            inv=set(fval(r, 'prism.hadamard.inverse_weight_names') or []),
                            tied=bool(fval(r, 'prism.hadamard.tied_output', False)),
                            gdn_grouped=bool(fval(r, 'prism.hadamard.gdn_v_grouped', False)))
        self._rots = {}
        T = {t.name: t for t in r.tensors}
        self.recr = [f'blk.{i}.ssm_a' in T for i in range(self.n_layer)]      # linear-attention (GDN) layers
        self.n_tern = self.n_train = 0

        def rot_for(name, width, perm=None, inverse=False):
            h = self.had
            if h is None or (not inverse and name not in h['names']) or (inverse and name not in h['inv']):
                return None
            key = (width, perm, inverse)
            if key not in self._rots:
                self._rots[key] = Rot(h['H'], h['signs'].get(width) if h['signs'] else None, perm, inverse)
                assert not h['signs'] or width in h['signs'], f'no sign vector for width {width}'
            return self._rots[key]

        def dense_np(t):
            ne = [int(x) for x in t.shape]
            if t.tensor_type in (0, 1):          # F32, F16
                a = np.array(t.data, dtype=np.float32)
            elif t.tensor_type == 30:            # BF16
                a = (np.array(t.data).view(np.uint16).astype(np.uint32) << 16).view(np.float32)
            else:
                a = dequantize(t.data, t.tensor_type)
            return a.reshape(list(reversed(ne)))

        def lin(name, perm=None, trainable=True):
            t = T[name]
            out_f, in_f = int(t.shape[1]), int(t.shape[0])
            rot = rot_for(name, in_f, perm)
            if int(t.tensor_type) == PQ2_0:
                pk, sc = pq2_split(np.array(t.data), out_f, in_f)
                pk = torch.from_numpy(pk).to(device)
                assert not bool(((pk & 0x03) == 3).any() | ((pk & 0x0C) == 12).any() | ((pk & 0x30) == 48).any()
                                | ((pk & 0xC0) == 192).any()), f'{name}: code 11 (+2) present, not ternary'
                m = TLinear(name, pk, torch.from_numpy(sc).to(device), rot, trainable, self.hp)
                self.n_tern += out_f * in_f
                self.n_train += out_f * in_f if trainable else 0
                return m
            return Dense(torch.from_numpy(dense_np(t)).to(device, dtype), rot)

        def vec(name):
            return torch.from_numpy(dense_np(T[name]).astype(np.float32)).to(device)

        # embedding (frozen by default) and output head
        te = T['token_embd.weight']
        self.vocab = int(te.shape[1])
        if int(te.tensor_type) == PQ2_0:
            pk, sc = pq2_split(np.array(te.data), self.vocab, self.n_embd)
            E = TLinear('token_embd.weight', torch.from_numpy(pk).to(device), torch.from_numpy(sc).to(device), None, False, self.hp)
            self.E = E.weight(dtype)
            self.n_tern += self.vocab * self.n_embd
            del E
        else:
            self.E = torch.from_numpy(dense_np(te)).to(device, dtype)
        self.embed_rot = rot_for('token_embd.weight', self.n_embd, inverse=True)
        if 'output.weight' in T:
            self.head = lin('output.weight', trainable=False)
        else:
            self.head = None
            tied = self.had is not None and self.had['tied']
            self.head_rot = Rot(self.had['H'], self.had['signs'].get(self.n_embd) if self.had['signs'] else None) if tied else None
        self.out_norm = vec('output_norm.weight')
        self.layers = []
        for i in range(self.n_layer):
            p = f'blk.{i}.'
            L = dict(recr=self.recr[i], attn_norm=vec(p + 'attn_norm.weight'), post_norm=vec(p + 'post_attention_norm.weight'),
                     ffn_gate=lin(p + 'ffn_gate.weight'), ffn_up=lin(p + 'ffn_up.weight'), ffn_down=lin(p + 'ffn_down.weight'))
            if self.recr[i]:
                perm = (self.hv, self.nk, self.nv // self.nk) if (self.had and self.had['gdn_grouped']) else None
                L.update(wqkv=lin(p + 'attn_qkv.weight'), wgate=lin(p + 'attn_gate.weight'),
                         beta=lin(p + 'ssm_beta.weight'), alpha=lin(p + 'ssm_alpha.weight'),
                         conv=vec(p + 'ssm_conv1d.weight'), dt=vec(p + 'ssm_dt.bias'), a=vec(p + 'ssm_a'),
                         ssm_norm=vec(p + 'ssm_norm.weight'), ssm_out=lin(p + 'ssm_out.weight', perm=perm))
            else:
                if p + 'attn_q.weight' in T:
                    L.update(wq=lin(p + 'attn_q.weight'), wk=lin(p + 'attn_k.weight'), wv=lin(p + 'attn_v.weight'))
                else:
                    L.update(wqkv=lin(p + 'attn_qkv.weight'))
                L.update(wo=lin(p + 'attn_output.weight'), q_norm=vec(p + 'attn_q_norm.weight'), k_norm=vec(p + 'attn_k_norm.weight'))
            self.layers.append(L)
            for k, m in L.items():
                if isinstance(m, nn.Module):
                    self.add_module(f'l{i}_{k}', m)
        del r
        self.tlayers = [m for m in self.modules() if isinstance(m, TLinear) and m.trainable]
        self._rope = {}

    # -- pieces
    def rope_cs(self, T):
        if self._rope.get('T', 0) < T:
            T0 = T
            T = max(T, 4096)
            n2 = self.n_rot // 2
            sec = self.sections
            sd = sum(sec)
            comp = []                            # which position component each rotated pair uses (ggml_mrope_cache_init)
            for i in range(n2):
                if sd == 0:
                    comp.append(0); continue
                s = i % sd
                if s % 3 == 1 and s < 3 * sec[1]: comp.append(1)
                elif s % 3 == 2 and s < 3 * sec[2]: comp.append(2)
                elif s % 3 == 0 and s < 3 * sec[0]: comp.append(0)
                else: comp.append(3)
            pos = torch.arange(T, device=self.device, dtype=torch.float32)
            pmat = torch.stack([pos, pos, pos, torch.zeros_like(pos)], 1)       # text: t = h = w = position, e = 0
            p = pmat[:, comp]                                                    # [T, n2]
            inv = self.base ** (-torch.arange(n2, device=self.device, dtype=torch.float32) * 2 / self.n_rot)
            ang = p * inv
            self._rope = dict(T=T, cs=(ang.cos(), ang.sin()))
            T = T0
        c, s = self._rope['cs']
        return c[:T], s[:T]

    def rope(self, x):                           # x [T, H, D], NEOX pairs (i, i + n_rot/2) on the first n_rot dims
        T = x.shape[0]
        cos, sin = self.rope_cs(T)
        n2 = self.n_rot // 2
        xf = x.float()
        x1, x2 = xf[..., :n2], xf[..., n2:self.n_rot]
        c, s = cos[:, None, :], sin[:, None, :]
        out = torch.cat([x1 * c - x2 * s, x2 * c + x1 * s, xf[..., self.n_rot:]], -1)
        return out.to(x.dtype)

    @staticmethod
    def shared_rot(x, mods):
        rots = [m.rot for m in mods if m.rot is not None]
        if rots and all(r is rots[0] for r in rots):
            return rots[0](x)
        return None

    def attn(self, L, x):
        T = x.shape[0]
        if 'wq' in L:
            xr = self.shared_rot(x, [L['wq'], L['wk'], L['wv']])
            qg, k, v = L['wq'](x, xr), L['wk'](x, xr), L['wv'](x, xr)
        else:
            qkv = L['wqkv'](x)
            nq = self.n_head * self.hd * 2
            qg, k, v = qkv[:, :nq], qkv[:, nq:nq + self.n_kv * self.hd], qkv[:, nq + self.n_kv * self.hd:]
        qg = qg.view(T, self.n_head, 2 * self.hd)
        q, gate = qg[..., :self.hd], qg[..., self.hd:].reshape(T, self.n_head * self.hd)
        q = self.rope(rms(q, L['q_norm'], self.eps))
        k = self.rope(rms(k.view(T, self.n_kv, self.hd), L['k_norm'], self.eps))
        v = v.view(T, self.n_kv, self.hd)
        rep = self.n_head // self.n_kv
        k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        o = F.scaled_dot_product_attention(q.transpose(0, 1)[None], k.transpose(0, 1)[None], v.transpose(0, 1)[None],
                                           is_causal=True, scale=self.hd ** -0.5)
        o = o[0].transpose(0, 1).reshape(T, self.n_head * self.hd)
        return L['wo'](o * torch.sigmoid(gate))

    def gdn_layer(self, L, x):
        T = x.shape[0]
        xr = self.shared_rot(x, [L['wqkv'], L['wgate']])
        qkv, z = L['wqkv'](x, xr), L['wgate'](x, xr)
        beta = torch.sigmoid(L['beta'](x).float())
        g = F.softplus(L['alpha'](x).float() + L['dt']) * L['a']
        C = qkv.shape[-1]
        w = L['conv'].reshape(C, 1, self.d_conv).to(qkv.dtype)
        c = F.conv1d(F.pad(qkv.t()[None], (self.d_conv - 1, 0)), w, groups=C)[0].t()
        c = F.silu(c)
        nkd = self.hk * self.nk
        q, k, v = c[:, :nkd].view(T, self.nk, self.hk), c[:, nkd:2 * nkd].view(T, self.nk, self.hk), c[:, 2 * nkd:].view(T, self.nv, self.hv)
        q = (q.float() / q.float().norm(dim=-1, keepdim=True).clamp(min=self.eps)).to(x.dtype)
        k = (k.float() / k.float().norm(dim=-1, keepdim=True).clamp(min=self.eps)).to(x.dtype)
        if self.nv != self.nk:                   # tiled: v head j uses k head j % nk (ggml_repeat / fused kernel)
            q, k = q.repeat(1, self.nv // self.nk, 1), k.repeat(1, self.nv // self.nk, 1)
        o = gdn(q[None].contiguous(), k[None].contiguous(), v[None].contiguous(), g[None], beta[None], self.hk ** -0.5)[0]
        o = rms(o, L['ssm_norm'], self.eps) * F.silu(z.view(T, self.nv, self.hv))
        return L['ssm_out'](o.reshape(T, self.nv * self.hv))

    def block(self, i, h):
        L = self.layers[i]
        x = rms(h, L['attn_norm'], self.eps)
        h = h + (self.gdn_layer(L, x) if L['recr'] else self.attn(L, x))
        x = rms(h, L['post_norm'], self.eps)
        xr = self.shared_rot(x, [L['ffn_gate'], L['ffn_up']])
        return h + L['ffn_down'](F.silu(L['ffn_gate'](x, xr)) * L['ffn_up'](x, xr))

    def hidden(self, ids, ckpt=False):
        h = self.E[ids]
        if self.embed_rot is not None:
            h = self.embed_rot(h)
        for i in range(self.n_layer):
            h = checkpoint(self.block, i, h, use_reentrant=False) if ckpt else self.block(i, h)
        return rms(h, self.out_norm, self.eps)

    def logits(self, h):
        if self.head is not None:
            return self.head(h).float()
        x = self.head_rot(h) if self.head_rot is not None else h
        return (x @ self.E.t()).float()

    def loss(self, ids, labels, ckpt=True, chunk=1024):
        """Mean cross-entropy over positions with labels != -100 (labels already shifted: labels[t] = ids[t+1])."""
        h = self.hidden(ids, ckpt)
        n = int((labels != -100).sum())
        tot = h.new_zeros((), dtype=torch.float32)
        for s in range(0, h.shape[0], chunk):
            lab = labels[s:s + chunk]
            if bool((lab != -100).any()):
                f = lambda hh, ll: F.cross_entropy(self.logits(hh), ll, ignore_index=-100, reduction='sum')
                tot = tot + (checkpoint(f, h[s:s + chunk], lab, use_reentrant=False) if ckpt else f(h[s:s + chunk], lab))
        return tot / max(n, 1), n

    @torch.no_grad()
    def greedy(self, ids, n_new, stop=()):
        ids = list(ids)
        for _ in range(n_new):
            t = torch.tensor(ids, device=self.device)
            nxt = int(self.logits(self.hidden(t)[-1:]).argmax(-1))
            ids.append(nxt)
            if nxt in stop:
                break
        return ids

    # -- bookkeeping
    def set_voting(self, on):
        for m in self.tlayers:
            m.voting = on

    def state_bytes(self):
        return sum(m.counter.numel() + m.packed.numel() for m in self.tlayers)

    def flips(self):
        return sum(m.flips for m in self.tlayers)

    def save_codes(self, path):
        torch.save({m.name: m.packed.cpu() for m in self.tlayers}, path + '.tmp')
        os.replace(path + '.tmp', path)

    def load_codes(self, path):
        d = torch.load(path)
        for m in self.tlayers:
            m.packed.copy_(d[m.name].to(m.packed.device))

    def export_gguf(self, out_path):
        """Copy the GGUF and overwrite the code bytes of every trainable tensor (scales unchanged)."""
        from gguf import GGUFReader
        shutil.copyfile(self.path, out_path)
        r = GGUFReader(self.path)
        T = {t.name: t for t in r.tensors}
        mm = np.memmap(out_path, dtype=np.uint8, mode='r+')
        changed = total = 0
        for m in self.tlayers:
            t = T[m.name]
            blocks = mm[t.data_offset:t.data_offset + t.n_bytes].reshape(m.out_f, m.in_f // QK, 2 + QK // 4)
            new = m.packed.cpu().numpy().reshape(m.out_f, m.in_f // QK, QK // 4)
            old = torch.from_numpy(np.ascontiguousarray(blocks[..., 2:]).reshape(m.out_f, -1))
            changed += int((unpack(old) != unpack(m.packed.cpu())).sum()); total += m.out_f * m.in_f
            blocks[..., 2:] = new
        mm.flush(); del mm
        r2 = GGUFReader(out_path)                 # read back and compare
        T2 = {t.name: t for t in r2.tensors}
        for m in self.tlayers[:3] + self.tlayers[-3:]:
            pk, _ = pq2_split(np.array(T2[m.name].data), m.out_f, m.in_f)
            assert np.array_equal(pk, m.packed.cpu().numpy()), f'export mismatch in {m.name}'
        return changed, total


# ----------------------------------------------------------------------------- data
ASSIST = re.compile(r'<\|im_start\|>assistant\n(.*?<\|im_end\|>)', re.S)


def norm_messages(msgs):
    out = []
    for m in msgs:
        m = {k: v for k, v in m.items() if v is not None}
        m.setdefault('content', '')
        if m.get('tool_calls'):
            tcs = []
            for tc in m['tool_calls']:
                tc = dict(tc)
                fn = dict(tc.get('function', tc))
                a = fn.get('arguments')
                if isinstance(a, str):
                    try:
                        fn['arguments'] = json.loads(a)
                    except Exception:
                        pass
                tcs.append({'type': 'function', 'function': {'name': fn.get('name'), 'arguments': fn.get('arguments', {})}})
            m['tool_calls'] = tcs
        out.append(m)
    return out


def encode(tok, ex, max_len):
    """Chat-template the trajectory; train only on assistant turns. Returns (ids, labels) or None."""
    try:
        text = tok.apply_chat_template(norm_messages(ex['messages']), tools=ex.get('tools') or None, tokenize=False)
    except Exception:
        return None
    enc = tok(text, return_offsets_mapping=True, add_special_tokens=False)
    ids, offs = enc['input_ids'][:max_len + 1], enc['offset_mapping'][:max_len + 1]
    spans = [(m.start(1), m.end(1)) for m in ASSIST.finditer(text)]
    tgt = np.zeros(len(ids), dtype=bool)
    j = 0
    for i, (a, _) in enumerate(offs):
        while j < len(spans) and spans[j][1] <= a:
            j += 1
        if j < len(spans) and spans[j][0] <= a < spans[j][1]:
            tgt[i] = True
    labels = [ids[i + 1] if tgt[i + 1] else -100 for i in range(len(ids) - 1)]
    if sum(l != -100 for l in labels) < 8:
        return None
    return ids[:-1], labels


def build_data(tok, configs, n_train, n_eval, max_len, seed=0):
    from datasets import load_dataset
    train, ev, stats = [], [], {}
    for cfg in configs:
        ds = load_dataset('openbmb/UltraData-SFT-Agent-2609', cfg, split='train', streaming=True)
        ds = ds.shuffle(seed=seed, buffer_size=5000)
        want_tr, want_ev = n_train // len(configs), n_eval // len(configs)
        a, b, skipped = [], [], 0
        for ex in ds:
            e = encode(tok, ex, max_len)
            if e is None:
                skipped += 1; continue
            (b if len(b) < want_ev else a).append(e)
            if len(a) >= want_tr and len(b) >= want_ev:
                break
        stats[cfg] = dict(train=len(a), eval=len(b), skipped=skipped,
                          mean_tokens=float(np.mean([len(x[0]) for x in a])), mean_targets=float(np.mean([sum(l != -100 for l in x[1]) for x in a])))
        train += a; ev += b
    rng = np.random.default_rng(seed)
    rng.shuffle(train)
    return train, ev, stats


@torch.no_grad()
def eval_loss(model, data, device):
    tot = n = 0
    for ids, lab in data:
        l, k = model.loss(torch.tensor(ids, device=device), torch.tensor(lab, device=device), ckpt=False)
        tot += float(l) * k; n += k
    return tot / max(n, 1)


@torch.no_grad()
def ppl_llamacpp_style(model, ids, n_ctx=512, n_chunks=16):
    """llama-perplexity: independent chunks of n_ctx tokens, score the second half of each."""
    nll = cnt = 0
    first = n_ctx // 2
    for c in range(min(n_chunks, len(ids) // n_ctx)):
        t = torch.tensor(ids[c * n_ctx:(c + 1) * n_ctx], device=model.device)
        lg = model.logits(model.hidden(t))
        lp = torch.log_softmax(lg[first:n_ctx - 1], -1)
        tgt = t[first + 1:n_ctx]
        nll -= float(lp.gather(1, tgt[:, None]).sum()); cnt += len(tgt)
    return math.exp(nll / cnt)

In [ ]:
%%writefile /content/make_tiny_bonsai.py
"""Write a tiny random Ternary-Bonsai-2-style GGUF (arch qwen35, PQ2_0 weights, Hadamard folding with explicit signs,
GDN v-head grouping, tied output, version-2 contract). Used to check the PyTorch port against PrismML's llama.cpp.
Usage: python make_tiny_bonsai.py OUT.gguf   (needs the fork's gguf-py on the path)"""
import sys
import numpy as np
import gguf
from gguf import GGUFWriter, GGMLQuantizationType as Q

out = sys.argv[1]
rng = np.random.default_rng(1234)
E, NL, NH, NKV, HD, NROT = 256, 4, 2, 1, 128, 32
HK, NK, NV, KC, FF, B = 64, 2, 4, 4, 512, 64
INNER = HK * NV
SECTIONS = [6, 5, 5, 0]
QK = 128


def pq2(rows, cols, scale):
    """random ternary codes + fp16 group scales, laid out as PQ2_0 blocks: bytes [rows, cols/128 * 34]"""
    codes = rng.choice([-1, 0, 1], size=(rows, cols), p=[0.3, 0.4, 0.3])
    v = (codes + 1).astype(np.uint8).reshape(rows, cols // QK, QK // 4, 4)
    qs = v[..., 0] | (v[..., 1] << 2) | (v[..., 2] << 4) | (v[..., 3] << 6)
    d = (scale * (0.5 + rng.random((rows, cols // QK)))).astype(np.float16)
    blocks = np.concatenate([d.view(np.uint8).reshape(rows, cols // QK, 2), qs], -1)
    return blocks.reshape(rows, -1)


w = GGUFWriter(out, 'qwen35')
w.add_string('general.name', 'tiny-bonsai-test')
A = 'qwen35.'
for k, v in dict(context_length=4096, embedding_length=E, block_count=NL, feed_forward_length=FF,
                 full_attention_interval=4).items():
    w.add_uint32(A + k, v)
w.add_uint32(A + 'attention.head_count', NH); w.add_uint32(A + 'attention.head_count_kv', NKV)
w.add_uint32(A + 'attention.key_length', HD); w.add_uint32(A + 'attention.value_length', HD)
w.add_float32(A + 'attention.layer_norm_rms_epsilon', 1e-6)
w.add_uint32(A + 'rope.dimension_count', NROT); w.add_float32(A + 'rope.freq_base', 10000.0)
w.add_array(A + 'rope.dimension_sections', SECTIONS)
w.add_uint32(A + 'ssm.conv_kernel', KC); w.add_uint32(A + 'ssm.inner_size', INNER)
w.add_uint32(A + 'ssm.state_size', HK); w.add_uint32(A + 'ssm.time_step_rank', NV); w.add_uint32(A + 'ssm.group_count', NK)
# byte-level vocab: 256 GPT-2 byte symbols + one special
bs = list(range(ord('!'), ord('~') + 1)) + list(range(ord('¡'), ord('¬') + 1)) + list(range(ord('®'), ord('ÿ') + 1))
cs, n = bs[:], 0
for b in range(256):
    if b not in bs:
        bs.append(b); cs.append(256 + n); n += 1
byte2uni = dict(zip(bs, [chr(c) for c in cs]))
toks = [byte2uni[b] for b in range(256)] + ['<|endoftext|>']
w.add_string('tokenizer.ggml.model', 'gpt2'); w.add_string('tokenizer.ggml.pre', 'default')
w.add_array('tokenizer.ggml.tokens', toks); w.add_array('tokenizer.ggml.token_type', [1] * 256 + [3])
w.add_array('tokenizer.ggml.merges', [byte2uni[ord('t')] + ' ' + byte2uni[ord('h')]][:0] or ['a b'])
w.add_uint32('tokenizer.ggml.bos_token_id', 256); w.add_uint32('tokenizer.ggml.eos_token_id', 256)
w.add_bool('tokenizer.ggml.add_bos_token', False)
V = len(toks)

names, tensors = [], []


def add(name, arr, raw=None):
    tensors.append((name, arr, raw))


def fold(name):
    names.append(name)


add('token_embd.weight', pq2(V, E, 0.15), Q.PQ2_0)
add('output_norm.weight', (1 + 0.1 * rng.standard_normal(E)).astype(np.float32))
for i in range(NL):
    p = f'blk.{i}.'
    add(p + 'attn_norm.weight', (1 + 0.1 * rng.standard_normal(E)).astype(np.float32))
    add(p + 'post_attention_norm.weight', (1 + 0.1 * rng.standard_normal(E)).astype(np.float32))
    for nm, (r, c) in dict(ffn_gate=(FF, E), ffn_up=(FF, E), ffn_down=(E, FF)).items():
        add(p + nm + '.weight', pq2(r, c, 1.5 / np.sqrt(c)), Q.PQ2_0); fold(p + nm + '.weight')
    if (i + 1) % 4 != 0:                                   # gated delta net layer
        add(p + 'attn_qkv.weight', pq2(2 * HK * NK + INNER, E, 1.5 / np.sqrt(E)), Q.PQ2_0); fold(p + 'attn_qkv.weight')
        add(p + 'attn_gate.weight', pq2(INNER, E, 1.5 / np.sqrt(E)), Q.PQ2_0); fold(p + 'attn_gate.weight')
        add(p + 'ssm_conv1d.weight', (0.4 * rng.standard_normal((2 * HK * NK + INNER, KC))).astype(np.float32))
        add(p + 'ssm_dt.bias', rng.standard_normal(NV).astype(np.float32))
        add(p + 'ssm_a', (-np.exp(0.5 * rng.standard_normal(NV))).astype(np.float32))
        add(p + 'ssm_beta.weight', (0.1 * rng.standard_normal((NV, E))).astype(np.float32))
        add(p + 'ssm_alpha.weight', (0.1 * rng.standard_normal((NV, E))).astype(np.float32))
        add(p + 'ssm_norm.weight', (1 + 0.1 * rng.standard_normal(HK)).astype(np.float32))
        add(p + 'ssm_out.weight', pq2(E, INNER, 1.5 / np.sqrt(INNER)), Q.PQ2_0); fold(p + 'ssm_out.weight')
    else:                                                  # full attention layer
        add(p + 'attn_q.weight', pq2(NH * HD * 2, E, 1.5 / np.sqrt(E)), Q.PQ2_0); fold(p + 'attn_q.weight')
        add(p + 'attn_k.weight', pq2(NKV * HD, E, 1.5 / np.sqrt(E)), Q.PQ2_0); fold(p + 'attn_k.weight')
        add(p + 'attn_v.weight', pq2(NKV * HD, E, 1.5 / np.sqrt(E)), Q.PQ2_0); fold(p + 'attn_v.weight')
        add(p + 'attn_output.weight', pq2(E, NH * HD, 1.5 / np.sqrt(NH * HD)), Q.PQ2_0); fold(p + 'attn_output.weight')
        add(p + 'attn_q_norm.weight', (1 + 0.1 * rng.standard_normal(HD)).astype(np.float32))
        add(p + 'attn_k_norm.weight', (1 + 0.1 * rng.standard_normal(HD)).astype(np.float32))

widths = sorted({E, FF, INNER, NH * HD})
w.add_uint32('prism.hadamard.version', 2); w.add_bool('prism.hadamard.tied_output', True)
w.add_uint32('prism.hadamard.block_size', B)
w.add_string('prism.hadamard.transform', 'normalized-sylvester-walsh-hadamard')
w.add_string('prism.hadamard.axis', 'input-last-dimension'); w.add_string('prism.hadamard.sign_mode', 'explicit')
w.add_array('prism.hadamard.weight_names', names)
w.add_array('prism.hadamard.inverse_weight_names', ['token_embd.weight'])
w.add_array('prism.hadamard.sign_widths', widths)
w.add_array('prism.hadamard.sign_values', [int(x) for wd in widths for x in rng.choice([-1, 1], wd)])
w.add_bool('prism.hadamard.gdn_v_grouped', True)
F32 = '--f32' in sys.argv            # same values as float32 tensors: llama.cpp then runs without activation quantization
for name, arr, raw in tensors:
    if F32 and raw == Q.PQ2_0:
        rows = arr.shape[0]
        bl = arr.reshape(rows, -1, 34)
        d = np.ascontiguousarray(bl[..., :2]).view(np.float16).astype(np.float32)
        q = np.stack([(bl[..., 2:] >> s) & 3 for s in (0, 2, 4, 6)], -1).reshape(rows, -1, QK).astype(np.float32) - 1
        w.add_tensor(name, (q * d).reshape(rows, -1))
    else:
        w.add_tensor(name, arr, raw_dtype=raw)
w.write_header_to_file(); w.write_kv_data_to_file(); w.write_tensors_to_file(); w.close()
print('wrote', out, len(tensors), 'tensors,', len(names), 'folded')

## Helpers

In [ ]:
import ast, importlib, json, math, re, time, torch, numpy as np
import bonsai_counter as bc
importlib.reload(bc)
BIN = '/content/llama.cpp/build/bin'
NT = str(os.cpu_count())
torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False

def llama_ppl(gguf, txt, ctx, chunks):
    r = subprocess.run([f'{BIN}/llama-perplexity', '-m', gguf, '-f', txt, '-c', str(ctx), '-b', str(ctx),
                        '--chunks', str(chunks), '-t', NT], capture_output=True, text=True)
    m = re.search(r'Final estimate: PPL = ([0-9.]+)', r.stdout + r.stderr)
    if not m: print((r.stdout + r.stderr)[-3000:]); raise RuntimeError('llama-perplexity failed')
    return float(m.group(1))

def llama_ids(gguf, txt):
    r = subprocess.run([f'{BIN}/llama-tokenize', '-m', gguf, '-f', txt, '--ids', '--log-disable'], capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith('[')]
    if not line: print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError('llama-tokenize failed')
    return ast.literal_eval(line[-1])

def llama_gen(gguf, prompt, n):
    r = subprocess.run([f'{BIN}/llama-simple', '-m', gguf, '-n', str(n), prompt], capture_output=True)
    speed = re.search(rb'speed: ([0-9.]+) t/s', r.stderr)
    return r.stdout, (float(speed.group(1)) if speed else None)

def gb(): return torch.cuda.max_memory_allocated() / 2**30

## Check 1: tiny model with the same architecture (~3 min)

In [ ]:
gdn_report = bc.pick_gdn('cuda')
TINY, TINY32, TXT = '/content/tiny.gguf', '/content/tiny_f32.gguf', '/content/tiny.txt'
env = dict(os.environ, PYTHONPATH='/content/llama.cpp/gguf-py')
subprocess.run([sys.executable, '/content/make_tiny_bonsai.py', TINY], check=True, env=env)
subprocess.run([sys.executable, '/content/make_tiny_bonsai.py', TINY32, '--f32'], check=True, env=env)   # same values, float32
open(TXT, 'w').write(''.join(f'Sample {i}: the quick brown fox jumps over the lazy dog while {i * 7 % 13} birds sing. ' for i in range(60)))
ids = llama_ids(TINY, TXT)
assert ids[:200] == list(open(TXT, 'rb').read()[:200]), 'byte tokenizer mismatch'
ref = llama_ppl(TINY32, TXT, 128, 3)          # float32 weights: llama.cpp runs without activation quantization
ref_q = llama_ppl(TINY, TXT, 128, 3)          # the PQ2_0 file as shipped (llama.cpp quantizes activations: ~0.2% off)
impl = bc.GDN_IMPL
m = bc.Bonsai(TINY, 'cuda', dtype=torch.float32)
bc.GDN_IMPL = 'naive'
ours = bc.ppl_llamacpp_style(m, ids, 128, 3)
bc.GDN_IMPL = impl
ours_k = bc.ppl_llamacpp_style(m, ids, 128, 3)
print(f'tiny model ppl: llama.cpp f32 {ref:.4f} (PQ2_0 file: {ref_q:.4f}) | ours, reference recurrence {ours:.4f} | ours, {impl} kernel {ours_k:.4f}')
assert abs(ours / ref - 1) < 1e-3, 'PORT MISMATCH: stop here and check the output above'
assert abs(ours_k / ref - 1) < 1e-2, 'KERNEL MISMATCH: stop here and check the output above'
prompt = 'Sample 3: the quick'
bc.GDN_IMPL = 'naive'
gen = bytes(m.greedy(list(prompt.encode()), 12)[len(prompt):])
bc.GDN_IMPL = impl
out, _ = llama_gen(TINY32, prompt, 14)
print('greedy ours :', gen); print('llama.cpp   :', out)
assert gen in out, 'GREEDY MISMATCH: stop here and check the output above'
# 30 counter steps on the tiny model, then export and reload in llama.cpp
m.hp.update(tau=2.0, vote=1.0, leak=0.0); m.set_voting(True)
seqs = [ids[i:i + 129] for i in range(0, 30 * 50, 50)]
first = last = 0
for s, sq in enumerate(seqs):
    t = torch.tensor(sq, device='cuda')
    l, _ = m.loss(t[:-1], t[1:])
    l.backward()
    first += float(l) if s < 5 else 0; last += float(l) if s >= 25 else 0
m.set_voting(False)
changed, total = m.export_gguf('/content/tiny_trained.gguf')
ref2, ours2 = llama_ppl('/content/tiny_trained.gguf', TXT, 128, 3), bc.ppl_llamacpp_style(m, ids, 128, 3)
print(f'tiny training: loss {first / 5:.3f} -> {last / 5:.3f}, {m.flips()} flips, {changed} of {total} weights changed')
print(f'exported tiny model ppl: llama.cpp {ref2:.4f} | ours {ours2:.4f}')
assert m.flips() > 0 and changed > 0, 'no weight flipped'
assert abs(ours2 / ref2 - 1) < 1e-2, 'EXPORT MISMATCH: stop here and check the output above'
print('CHECK 1 PASSED'); del m; torch.cuda.empty_cache()

## Download Bonsai 2 9B (PQ2_0, 2.4 GB) and the Qwen3.5 tokenizer; check the tokenizer against llama.cpp

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoTokenizer
REPO = 'prism-ml/Ternary-Bonsai-2-9B-gguf'
files = HfApi().list_repo_files(REPO)
cand = [f for f in files if f.endswith('.gguf') and 'PQ2_0' in f and 'MTP' not in f]
print('PQ2_0 files:', cand)
GG = hf_hub_download(REPO, cand[0], local_dir='/content/bonsai')
tok = AutoTokenizer.from_pretrained('Qwen/Qwen3.5-9B')
sample = tok.apply_chat_template([{'role': 'user', 'content': 'Write a Python function that sorts a list. Ça marche? 123'}],
                                 tokenize=False, add_generation_prompt=True) + 'def f(xs):\n    return sorted(xs)\n'
open('/content/sample.txt', 'w').write(sample)
a, b = tok(sample, add_special_tokens=False)['input_ids'], llama_ids(GG, '/content/sample.txt')
print('tokens HF', len(a), '| llama.cpp', len(b), '| identical:', a == b)
assert a == b, 'TOKENIZER MISMATCH: stop here and check the output above'

## Check 2: the real model, our port vs llama.cpp (~10 min; llama.cpp runs on the CPU)

In [ ]:
from datasets import load_dataset
wt = '\n'.join(load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')['text'])
open('/content/wiki.txt', 'w').write(wt[:60000])
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model = bc.Bonsai(GG, 'cuda')
print(f'loaded in {time.time() - t0:.0f} s | layers {model.n_layer} ({sum(model.recr)} linear-attention) | '
      f'ternary weights {model.n_tern / 1e9:.2f} B, trainable {model.n_train / 1e9:.2f} B | '
      f'training state {model.state_bytes() / 2**30:.1f} GB | GPU {gb():.1f} GB | Hadamard: {model.had is not None}')
ids_wiki = llama_ids(GG, '/content/wiki.txt')
t0 = time.time(); ref = llama_ppl(GG, '/content/wiki.txt', 512, 8); tref = time.time() - t0
ours = bc.ppl_llamacpp_style(model, ids_wiki, 512, 8)
print(f'WikiText-2 ppl (8 x 512 tokens): llama.cpp CPU {ref:.3f} ({tref:.0f} s) | ours GPU bf16 {ours:.3f} | diff {100 * (ours / ref - 1):+.2f}%')
assert abs(ours / ref - 1) < 0.03, 'REAL-MODEL MISMATCH: stop here and check the output above'
for p in ['The capital of France is', 'def fibonacci(n):']:
    pid = tok(p, add_special_tokens=False)['input_ids']
    g = tok.decode(model.greedy(pid, 16)[len(pid):])
    out, sp = llama_gen(GG, p, 16)
    print(f'{p!r}\n  ours     : {g!r}\n  llama.cpp: {out.decode(errors="replace")!r}  ({sp} tok/s on CPU)')
# one training step on a long sequence: memory and speed
torch.cuda.reset_peak_memory_stats()
model.set_voting(True)
x = torch.tensor(ids_wiki[:4097], device='cuda')
t0 = time.time(); l, _ = model.loss(x[:-1], x[1:]); l.backward(); torch.cuda.synchronize()
print(f'one 4096-token training step: {time.time() - t0:.1f} s, peak GPU {gb():.1f} GB, flips {model.flips()} (expected 0 at tau 32)')
model.set_voting(False)
if model.flips():      # a single flip in a long test step is possible: reload so training starts from the exact original
    print(f'{model.flips()} weight(s) flipped in the test step: reloading the original model')
    del model, l; torch.cuda.empty_cache(); model = bc.Bonsai(GG, 'cuda')
for mm in model.tlayers: mm.counter.zero_()
assert model.flips() == 0
print('CHECK 2 PASSED')

## Data: Tool-Use + Code-Agent trajectories (~5–10 min, cached on Drive)

In [ ]:
import pickle
N_TRAIN, N_EVAL, MAX_LEN = 3000, 100, 4096
DATA = f'{W}/data_{N_TRAIN}_{N_EVAL}_{MAX_LEN}.pkl'
if os.path.exists(DATA):
    train, ev, dstats = pickle.load(open(DATA, 'rb'))
else:
    train, ev, dstats = bc.build_data(tok, ['Tool-Use', 'Code-Agent'], N_TRAIN, N_EVAL, MAX_LEN)
    pickle.dump((train, ev, dstats), open(DATA, 'wb'))
print(json.dumps(dstats, indent=1))
print(f'train {len(train)} trajectories, {sum(len(x[0]) for x in train) / 1e6:.1f} M tokens; eval {len(ev)}')
# what the model is trained on: the first trajectory, target tokens only
ids0, lab0 = train[0]
print(tok.decode([t for t in lab0 if t != -100])[:600])

## Before training: held-out loss on agent data + WikiText perplexity (forgetting baseline)

In [ ]:
RUN = 'bonsai2_9b_tooluse_codeagent_t32'
OUT = f'{W}/runs/{RUN}'; os.makedirs(OUT, exist_ok=True)
res_path = f'{OUT}/result.json'
res = json.load(open(res_path)) if os.path.exists(res_path) else {}
if 'before' not in res:
    t0 = time.time()
    res['before'] = dict(eval_loss=bc.eval_loss(model, ev, 'cuda'), wiki_ppl=bc.ppl_llamacpp_style(model, ids_wiki, 512, 16))
    res['gdn'] = bc.GDN_IMPL; res['data'] = dstats
    json.dump(res, open(res_path, 'w'), indent=1)
    print(f'({time.time() - t0:.0f} s)')
print('before:', res['before'])

## Train (time budget; progress every 25 steps; codes saved to Drive every 30 min, rerun this cell to resume)

In [ ]:
HOURS = 2.5
model.hp.update(tau=32.0, vote=0.1, leak=0.003)
ck = f'{OUT}/codes.pt'
done = res.get('steps_done', 0)
if done and os.path.exists(ck):
    model.load_codes(ck); print(f'resumed after {done} steps (counters restart at 0)')
model.set_voting(True)
torch.cuda.reset_peak_memory_stats()
t0 = last_save = time.time(); hist, toks = [], 0
step = done
while step < len(train) and time.time() - t0 < HOURS * 3600:
    ids, lab = train[step]
    l, _ = model.loss(torch.tensor(ids, device='cuda'), torch.tensor(lab, device='cuda'))
    l.backward()
    for p in model.parameters(): p.grad = None
    hist.append(float(l)); toks += len(ids); step += 1
    if step % 25 == 0:
        el = time.time() - t0
        print(f'[{el / 60:5.1f} min] step {step} | loss (last 25) {np.mean(hist[-25:]):.4f} | flipped '
              f'{100 * model.flips() / model.n_train:.5f}% | {toks / el:,.0f} tok/s | peak {gb():.1f} GB', flush=True)
    if time.time() - last_save > 1800:
        model.save_codes(ck); res['steps_done'] = step; json.dump(res, open(res_path, 'w'), indent=1); last_save = time.time()
model.set_voting(False)
model.save_codes(ck)
res.update(steps_done=step, train_hours=res.get('train_hours', 0) + (time.time() - t0) / 3600, train_tokens=res.get('train_tokens', 0) + toks,
           train_loss_curve=res.get('train_loss_curve', []) + [float(np.mean(hist[i:i + 25])) for i in range(0, len(hist), 25)],
           peak_gb=gb(), state_gb=model.state_bytes() / 2**30, flips=model.flips())
json.dump(res, open(res_path, 'w'), indent=1)
print(f'done: {step} steps, {res["train_tokens"] / 1e6:.1f} M tokens, {res["train_hours"]:.2f} h')

## After training: same evaluation, export to GGUF, check and demo it in llama.cpp on the CPU

In [ ]:
res['after'] = dict(eval_loss=bc.eval_loss(model, ev, 'cuda'), wiki_ppl=bc.ppl_llamacpp_style(model, ids_wiki, 512, 16))
GOUT = f'/content/{RUN}.gguf'
changed, total = model.export_gguf(GOUT)
res['weights_changed'] = changed; res['weights_changed_pct'] = 100 * changed / total
ref = llama_ppl(GOUT, '/content/wiki.txt', 512, 8); ours = bc.ppl_llamacpp_style(model, ids_wiki, 512, 8)
res['export_check'] = dict(llama_cpp_ppl=ref, ours_ppl=ours)
json.dump(res, open(res_path, 'w'), indent=1)
b, a = res['before'], res['after']
print(f"held-out agent loss : {b['eval_loss']:.4f} -> {a['eval_loss']:.4f} ({a['eval_loss'] - b['eval_loss']:+.4f})")
print(f"WikiText-2 ppl      : {b['wiki_ppl']:.3f} -> {a['wiki_ppl']:.3f} ({100 * (a['wiki_ppl'] / b['wiki_ppl'] - 1):+.1f}%)")
print(f"weights changed     : {changed:,} of {total:,} ({res['weights_changed_pct']:.4f}%)")
print(f"exported GGUF in llama.cpp: ppl {ref:.3f} vs ours {ours:.3f}")
# demo: the shortest held-out tool-use prompt, original vs fine-tuned, both in llama.cpp on the CPU
exd = min(ev, key=lambda e: len(e[0]))
first_t = next(i for i, l in enumerate(exd[1]) if l != -100) + 1
prompt = tok.decode(exd[0][:first_t])
for name, g in (('original', GG), ('fine-tuned', GOUT)):
    out, sp = llama_gen(g, prompt, 160)
    print(f'\n===== {name} ({sp} tok/s on {NT} CPU threads) =====\n' + out.decode(errors='replace')[len(prompt) - 200:])
print('\nreference answer:\n' + tok.decode([t for t in exd[1][first_t - 1:] if t != -100])[:500])
!cp {GOUT} "{OUT}/"
print('saved', f'{OUT}/{RUN}.gguf')